# Qo‘sh va Bog‘lash akslantirishlarining tezligini solishtirish

Faqat ikkita akslantirish solishtiriladi (algoritmning qolgan qismisiz):
- **Qo‘sh(holat, X)** — standartdagi akslantirish (6.2.1-band);
- **Bog‘lash(holat, X)** — uning o‘rniga taklif etilgan yagona akslantirish (256 va 128 bitli rejimlar uchun bir xil).

Usullar: (1) C realizatsiyasi (gcc -O2) — asosiy o‘lchov; (2) Python (numpy) realizatsiyasi; (3) amallar soni.
Daftar mustaqil ishlaydi, faqat `numpy` kerak; C o‘lchovi uchun gcc/clang (Windows da MinGW) kerak.

In [1]:
# 0-hujayra. Dastur modullarini yuklash (daftar mustaqil ishlaydi).
# Agar modullar shu papkada bo'lsa, ular ishlatiladi; bo'lmasa quyidagi matndan xotirada yaratiladi.
import sys, types, importlib.util
_MODULLAR = {
    'algoritm': '"""\nO‘zMSt 285:2024, 1-algoritm: vektorlashtirilgan realizatsiya.\nVariantlar: p = 256 yoki p = 16; yutish o‘zgartirishi bz = \'std\' (BaytZichlash) yoki \'sbz\' (S(h ⊕ S(n))).\nBarcha funksiyalar (N, 32) o‘lchamli numpy massivlar ustida ishlaydi: bir vaqtda N ta mustaqil hisob.\n"""\nimport numpy as np\n\n# B ilova (majburiy): 1x256 chiziqli zichlash massivi (satr — yuqori yarim bayt)\nZICHLASH = bytes.fromhex(\n    "050d060b010a0f0800040709020c030e"\n    "0807020e0f030b06010c0d0a05040900"\n    "0e020d040c07010b06090005030a080f"\n    "000e090c030d07040f0605010b020a08"\n    "030a0702040c09010e0d0f0800050b06"\n    "02030108000e05090c0b06070a0f0d04"\n    "0a040e0f090508020b0001030c06070d"\n    "0b090a0106040d0f03050e000807020c"\n    "010003070d0b0a0c090e04060f080502"\n    "04080b090e0602050a030c0f070d0001"\n    "090c0f0002010e0a05080b0d04030607"\n    "060b080d07090003040f0a020e010c05"\n    "0f0100050a0803070d02090c060e040b"\n    "0c05040a0b02060d0807030e01000f09"\n    "070f0c060500040e020a080b0d090103"\n    "0d060503080f0c0007010204090b0e0a")\nZ = np.array(list(ZICHLASH), dtype=np.int64)\ndef aes_sbox():\n    s=[0]*256; p=q=1\n    while True:\n        p ^= (p<<1) ^ (0x1b if p&0x80 else 0); p&=0xff\n        q ^= q<<1; q ^= q<<2; q ^= q<<4; q&=0xff\n        if q&0x80: q^=0x09\n        x=q ^ ((q<<1|q>>7)&0xff) ^ ((q<<2|q>>6)&0xff) ^ ((q<<3|q>>5)&0xff) ^ ((q<<4|q>>4)&0xff)\n        s[p]=x^0x63\n        if p==1: break\n    s[0]=0x63; return np.array(s,dtype=np.int64)\nS=aes_sbox()\nclass Cfg:\n    def __init__(s,p=256,bz=\'sbz\'):\n        s.p=p; s.bz=bz\n        s.INV=np.zeros(p,dtype=np.int64)\n        for x in range(1,p,2): s.INV[x]=pow(x,-1,p)\ndef prododd(a,p):\n    b=np.where(a%2==0,a+1,a)%p; r=np.ones(a.shape[0],dtype=np.int64)\n    for i in range(a.shape[1]): r=(r*b[:,i])%p\n    return r\ndef dK(P,Jo,p): return np.where((P==(Jo+1)%p)|(P==(-(Jo+1))%p),10,0)\ndef step(x,A,B,R,c):\n    d=(B+R*x)%c.p; return (-A*x*c.INV[d])%c.p\ndef qosh(h,hn,R,c):\n    p=c.p; R=R[:,None]\n    J1h=h.sum(1)%p; J1hn=hn.sum(1)%p\n    Ph=prododd(h,p); Phn=prododd(hn,p)\n    K1h=(Ph+dK(Ph,J1hn,p))%p; K1hn=(Phn+dK(Phn,J1h,p))%p\n    dAh=np.where(J1hn%2==1,0,1); dAhn=np.where(J1h%2==1,0,1)\n    A1h=(J1hn[:,None]+2*hn+dAh[:,None])%p; A1hn=(J1h[:,None]+2*h+dAhn[:,None])%p\n    h1=step(h,A1h,(K1h[:,None]-2*h)%p,R,c); hn1=step(hn,A1hn,(K1hn[:,None]-2*hn)%p,R,c)\n    J2h=h1.sum(1)%p; J2hn=hn1.sum(1)%p\n    P2h=prododd(h1,p); P2hn=prododd(hn1,p)\n    K2h=(P2h+dK(P2h,J2hn,p))%p; K2hn=(P2hn+dK(P2hn,J2h,p))%p\n    dA2h=np.where((J1hn+J2hn)%2==1,0,1); dA2hn=np.where((J1h+J2h)%2==1,0,1)\n    A2h=(J1hn[:,None]+J2hn[:,None]+2*hn1+dA2h[:,None])%p; A2h=np.where(A2h==0,1,A2h)\n    A2hn=(J1h[:,None]+J2h[:,None]+2*h1+dA2hn[:,None])%p; A2hn=np.where(A2hn==0,1,A2hn)\n    B2h=((K1h*K2h)[:,None]-2*h1)%p; B2hn=((K1hn*K2hn)[:,None]-2*hn1)%p\n    h2=step(h1,A2h,B2h,R,c); h2=np.where(h2==0,(A2h+B2h)%p,h2)\n    hn2=step(hn1,A2hn,B2hn,R,c); hn2=np.where(hn2==0,(A2hn+B2hn)%p,hn2)\n    return step(h2,A2h,B2h,R,c), step(hn2,A2hn,B2hn,R,c)\ndef bzf(h,hn,c):\n    if c.p==256:\n        if c.bz==\'std\': return (Z[h]<<4)|Z[hn]\n        return S[h^S[hn]]\n    if c.bz==\'std\': return h^hn\n    H=16*h[:,0::2]+h[:,1::2]; N=16*hn[:,0::2]+hn[:,1::2]; Q=S[H^S[N]]\n    out=np.empty_like(h); out[:,0::2]=Q>>4; out[:,1::2]=Q&15; return out\ndef dia(H,K,p):  # H,K: (N,4,4)\n    sk=K.sum(1)            # sum_i K[i,u]  -> (N,4)\n    sh=H.sum(1)            # sum_i H[i,u]\n    R=np.empty_like(H)\n    for s in range(4):\n        for u in range(4):\n            if s==u:\n                v=H[:,u,u]*sk[:,u]-sum(H[:,i,i]*K[:,i,u] for i in range(4) if i!=u)\n            else:\n                v=H[:,s,u]*sk[:,u]+K[:,s,u]*sh[:,u]-sum(H[:,s,i]*K[:,i,u] for i in range(4) if i not in (s,u))\n            R[:,s,u]=v%p\n    return R\ndef aralash(h,ke,c):\n    H=h.reshape(-1,4,8); K=ke.reshape(-1,4,8)\n    L=dia(H[:,:,:4],K[:,:,:4],c.p); Rr=dia(H[:,:,4:],K[:,:,4:],c.p)\n    return np.concatenate([L,Rr],axis=2).reshape(-1,32)\ndef _shift(h,rows,cols):\n    m=h.reshape(-1,4,8).copy()\n    for s in range(4): m[:,s,:]=np.roll(m[:,s,:],rows[s],axis=1)\n    for u,n in cols.items(): m[:,:,u]=np.roll(m[:,:,u],n,axis=1)\n    return m.reshape(-1,32)\ndef surholat(h): return _shift(h,[1,2,3,4],{0:1,4:1,1:2,5:2,2:3,6:3})\ndef surkalit(ke): return _shift(ke,[4,3,2,1],{1:-3,5:-3,2:-2,6:-2,3:-1,7:-1})\ndef tuz(ke,k,c):\n    p=c.p\n    return np.where((ke==0)&(k%2==0),2,np.where((ke!=0)&(ke%2==k%2),ke,(ke-1)%p))\ndef iteration(holat,blok,ke,k,c,trace=None,rounds=10):\n    """Bitta tashqi iteratsiya (i): Qo\'sh+BZ, so\'ng \'rounds\' ta bosqich."""\n    R=k[:,0]\n    def T(name,j,h,kk):\n        if trace is not None: trace.append((name,j,h.copy(),kk.copy()))\n    holat,hn=qosh(holat,blok,R,c); T(\'Qosh0\',0,holat,ke)\n    holat=bzf(holat,hn,c); T(\'BZ0\',0,holat,ke)\n    for j in range(1,rounds+1):\n        holat=aralash(holat,ke,c); T(\'Aralash\',j,holat,ke)\n        holat,ke=qosh(holat,ke,R,c); T(\'Qosh1\',j,holat,ke)\n        holat=surholat(holat); ke=surkalit(ke); T(\'Sur\',j,holat,ke)\n        if j==8:\n            holat,hn=qosh(holat,blok,R,c); T(\'Qosh_inj8\',j,holat,ke); holat=bzf(holat,hn,c); T(\'BZ8\',j,holat,ke)\n        else:\n            holat,ke=qosh(holat,ke,R,c); T(\'Qosh2\',j,holat,ke)\n        ke=tuz(ke,k,c); T(\'Tuz\',j,holat,ke)\n        if j==9:\n            holat,hn=qosh(holat,blok,R,c); T(\'Qosh_inj9\',j,holat,ke); holat=bzf(holat,hn,c); T(\'BZ9\',j,holat,ke)\n    return holat,ke\ndef to_units(b,p):\n    b=np.asarray(b,dtype=np.int64)\n    if p==256: return b\n    out=np.empty(b.shape[:-1]+(b.shape[-1]*2,),dtype=np.int64); out[...,0::2]=b>>4; out[...,1::2]=b&15; return out\ndef hash_msgs(msgs,keys,c):\n    """msgs: (N,L) bayt, L - blok uzunligiga karrali; keys: (N,32 yoki 16)."""\n    return _hash_core(msgs,keys,c,bits=msgs.shape[1]*8)\ndef _hash_core(msgs,keys,c,bits):\n    N,L=msgs.shape; bl=32 if c.p==256 else 16\n    nb=L//bl\n    blocks=[msgs[:,i*bl:(i+1)*bl] for i in range(nb)]\n    ny=[0]*N\n    nyb=np.zeros((N,bl),dtype=np.int64)\n    for n in range(N):\n        v=sum(int.from_bytes(bytes(b[n].tolist()),\'big\') for b in blocks)%(2**(8*bl))\n        nyb[n]=list(v.to_bytes(bl,\'big\'))\n    lb=np.tile(np.array(list(bits.to_bytes(bl,\'big\'))),(N,1))\n    kir=[to_units(b,c.p) for b in blocks]+[to_units(lb,c.p),to_units(nyb,c.p)]\n    k=to_units(keys,c.p); ke=k.copy(); holat=kir[0]\n    for i in range(1,len(kir)): holat,ke=iteration(holat,kir[i],ke,k,c)\n    return holat\n\n\n# ------------------------------------------------------------------ nazorat qiymatlari\nKAT_KALIT = bytes.fromhex("c86ed4b2756df1cd6b6eca81d441809221ab81bba0c7c15044be8feb934f95a2")\nKAT_XABAR = b"funksiyaxeshirovaniyainformasionnayatexnologiyakriptograficheska"\nKAT = {\n    (256, \'std\'): "55394f2d84fccd4d8d01d8fef756cd657ebecc3f96f81e5207767447de66e2b6",\n    (256, \'sbz\'): "42f97daaa8e7010bc629ca695eaa77df7ae7443a0c3ae78d205c2e025d256c96",\n    (16, \'std\'):  "e9eaccdd4ccce2d229228c864989eb9f",\n    (16, \'sbz\'):  "bd745e2ed572ea4246ed68482ef46c5e",\n}\n\ndef xesh(xabar: bytes, kalit: bytes, p=256, bz=\'sbz\') -> bytes:\n    """Bitta xabar uchun xesh qiymat (xabar uzunligi blok uzunligiga karrali bo‘lishi shart emas)."""\n    bl = 32 if p == 256 else 16\n    m = xabar + b"\\x00" * (-len(xabar) % bl) if xabar else b"\\x00" * bl\n    # uzunlik bloki asl xabar uzunligi bo‘yicha hisoblanadi\n    c = Cfg(p, bz)\n    msgs = np.array([list(m)]); keys = np.array([list(kalit[:bl])])\n    out = _hash_core(msgs, keys, c, bits=len(xabar) * 8)[0]\n    if p == 16: out = [out[2*i] << 4 | out[2*i+1] for i in range(16)]\n    return bytes(int(x) for x in out)\n\ndef nazorat():\n    """A ilova va SBZ materialidagi nazorat qiymatlarini tekshiradi."""\n    ok = True\n    for (p, bz), kut in KAT.items():\n        r = xesh(KAT_XABAR, KAT_KALIT, p, bz).hex()\n        print(f"  p={p:3d}, {bz}: {\'MOS\' if r == kut else \'MOS EMAS\'}")\n        ok &= (r == kut)\n    return ok\n\nif __name__ == "__main__":\n    print("Nazorat qiymatlari:"); print("Hammasi mos" if nazorat() else "XATO")\n',
    'komponentlar': '"""Kriptotahlil uchun umumiy yordamchi funksiyalar."""\nimport itertools\nimport numpy as np\nfrom algoritm import Cfg, aralash, surholat\n\nPAR8 = np.array([bin(i).count(\'1\') & 1 for i in range(256)])\nX256 = np.arange(256)\n\ndef kalitlar(rng, N, p):\n    """Tasodifiy kalitlar: R (1-element) va 2-element juft (5.1.2-band talabi)."""\n    if p == 256:\n        k = rng.integers(0, 256, (N, 32)); k[:, 0] &= 0xFE; k[:, 1] &= 0xFE\n    else:\n        k = rng.integers(0, 16, (N, 32)); k[:, 0] &= 0xE; k[:, 1] &= 0xE\n    return k\n\ndef wht(f):\n    """Walsh–Adamar o‘zgartirishi (f — ±1 vektor)."""\n    a = f.astype(np.int64).copy(); h = 1; n = len(a)\n    while h < n:\n        a = a.reshape(-1, 2, h); a = np.stack([a[:, 0] + a[:, 1], a[:, 0] - a[:, 1]], 1).reshape(-1); h *= 2\n    return a\n\ndef lat(F, nout):\n    """8 bitli kirishli F uchun W[β][α] = Σ (−1)^(β·F(x) ⊕ α·x); korrelyatsiya c = W/256."""\n    W = np.zeros((1 << nout, 256), dtype=np.int64)\n    for b in range(1 << nout):\n        W[b] = wht(1 - 2 * PAR8[F & b])\n    return W\n\ndef ddt(F, nout):\n    """Ayirmalar matritsasi DDT[Δx][Δy] (8 bitli kirish)."""\n    return np.array([np.bincount(F ^ F[X256 ^ d], minlength=nout) for d in range(256)])\n\ndef anf_deg(fbits, nv):\n    """Boolean funksiyaning ANF darajasi (Möbius o‘zgartirishi)."""\n    a = fbits.astype(np.uint8).copy(); h = 1; N = 1 << nv\n    while h < N:\n        a = a.reshape(-1, 2, h); a[:, 1] ^= a[:, 0]; a = a.reshape(-1); h *= 2\n    wt = np.array([bin(i).count(\'1\') for i in range(N)]); nz = wt[a == 1]\n    return int(nz.max()) if len(nz) else -1\n\ndef bits(v, n): return (v[:, None] >> np.arange(n)) & 1\n\ndef implicit_count(points, deg):\n    """deg darajali chiziqli bog‘liq bo‘lmagan implitsit tenglamalar soni (monomlar matritsasi rangi orqali)."""\n    npts, nv = points.shape\n    mons = [()] + [m for d in range(1, deg + 1) for m in itertools.combinations(range(nv), d)]\n    words = (npts + 63) // 64; cols = []\n    for m in mons:\n        v = np.ones(npts, dtype=np.uint8)\n        for i in m: v &= points[:, i].astype(np.uint8)\n        pad = np.zeros(words * 64, dtype=np.uint8); pad[:npts] = v\n        cols.append(np.packbits(pad).view(np.uint64))\n    A = np.array(cols); rank = 0; n = A.shape[0]\n    for w in range(A.shape[1]):\n        for b in range(64):\n            if rank >= n: break\n            mask = np.uint64(1) << np.uint64(b)\n            idx = np.nonzero((A[rank:, w] & mask) != 0)[0]\n            if len(idx) == 0: continue\n            piv = rank + idx[0]\n            if piv != rank: A[[rank, piv]] = A[[piv, rank]]\n            rows = np.nonzero((A[:, w] & mask) != 0)[0]; rows = rows[rows != rank]\n            A[rows] ^= A[rank]; rank += 1\n    return len(mons) - rank, len(mons)\n\ndef qosh_elem(A1, B1, A2, B2, R, p=256):\n    """Qo‘sh o‘zgartirishining bitta elementi (parametrlar belgilangan): x → x^(−3)."""\n    x = np.arange(p); INV = np.zeros(p, dtype=np.int64)\n    for t in range(1, p, 2): INV[t] = pow(t, -1, p)\n    s = lambda x, A, B: (-A * x * INV[(B + R * x) % p]) % p\n    y1 = s(x, A1, B1); y2 = s(y1, A2, B2); y2 = np.where(y2 == 0, (A2 + B2) % p, y2)\n    return s(y2, A2, B2)\n\ndef tasodifiy_param(rng):\n    A1, B1, A2, B2 = [int(rng.integers(0, 128)) * 2 + 1 for _ in range(4)]\n    return A1, B1, A2, B2, int(rng.integers(0, 128)) * 2\n\ndef rank2(A):\n    A = A.copy() % 2; r = 0\n    for col in range(A.shape[1]):\n        piv = [i for i in range(r, A.shape[0]) if A[i, col]]\n        if not piv: continue\n        A[[r, piv[0]]] = A[[piv[0], r]]\n        for i in range(A.shape[0]):\n            if i != r and A[i, col]: A[i] ^= A[r]\n        r += 1\n    return r\n\n# 0-qatlam matritsalari: SurHolat (P) va Aralash (M(k0))\n_C2 = Cfg(2, \'std\')\n_perm = surholat(np.arange(32)[None, :].copy())[0]\nP_SUR = np.zeros((32, 32), dtype=np.int64); P_SUR[np.arange(32), _perm] = 1\n\ndef M_aralash(k):\n    """h0\' = M(k0)·h0 (mod 2) matritsasi."""\n    E = np.eye(32, dtype=np.int64); K = np.tile(np.asarray(k) % 2, (32, 1))\n    return (aralash(E, K, _C2).T) % 2\n\ndef L_raund(k):\n    """Yutishsiz raundning 0-qatlam matritsasi L = P·M(k0)."""\n    return (P_SUR @ M_aralash(k)) % 2\n',
    'boglash': '"""\nBog‘lash(holat, X) — Qo‘sh o‘rniga taklif etilayotgan yagona akslantirish (256 va 128 bitli rejimlar uchun bir xil).\nIkkala rejimda ham bayt darajasida ishlaydi: 128 bitli rejimda qo‘shni yarim baytlar juftlab baytga birlashtiriladi\n(SBZ dagi kabi), so‘ng natija yana yarim baytlarga ajratiladi.\n  Yarim qadam (T ← f(U)):  J = Σ U,  K = Π (U ∨ 1)  (mod 256)\n      aᵢ = 2·S(U[i] ⊕ X_U[i] ⊕ J) + 1,   bᵢ = 2·S(U[i+1] ⊕ X_T[i] ⊕ K) + 1\n      T[i] ← S( −aᵢ·T[i]·(bᵢ + R·T[i])⁻¹ ⊕ X_T[i] )        (R — kalitning birinchi bayti, juft)\n  Bog‘lash: o‘ng ← f(chap), chap ← f(o‘ng), o‘ng ← f(chap).\n"""\nimport numpy as np\nfrom algoritm import S, Cfg, aralash, surholat, surkalit, bzf, to_units\n\nINV = np.zeros(256, dtype=np.int64)\nfor _x in range(1, 256, 2): INV[_x] = pow(_x, -1, 256)\nSI = np.zeros(256, dtype=np.int64); SI[S] = np.arange(256)\n\ndef juftlash(u, p):      # birliklar -> baytlar\n    return u if p == 256 else 16 * u[:, 0::2] + u[:, 1::2]\ndef ajratish(b, p):      # baytlar -> birliklar\n    if p == 256: return b\n    u = np.empty((b.shape[0], 32), dtype=np.int64); u[:, 0::2] = b >> 4; u[:, 1::2] = b & 15; return u\ndef yarimlar(m):         # m = 32 yoki 16 bayt; satrda w bayt; chap yarim — satrning birinchi yarmi\n    w = 8 if m == 32 else 4\n    i = np.arange(m); return i[(i % w) < w // 2], i[(i % w) >= w // 2]\n\ndef _param(U, XT, XU):\n    J = U.sum(1) % 256; K = np.ones(U.shape[0], dtype=np.int64)\n    for i in range(U.shape[1]): K = (K * (U[:, i] | 1)) % 256\n    sig = np.roll(np.arange(U.shape[1]), -1)\n    a = (2 * S[U ^ XU ^ J[:, None]] + 1) % 256\n    b = (2 * S[U[:, sig] ^ XT ^ K[:, None]] + 1) % 256\n    return a, b\ndef _yarim(T, U, XT, XU, R):\n    a, b = _param(U, XT, XU)\n    return S[((-a * T * INV[(b + R[:, None] * T) % 256]) % 256) ^ XT]\ndef _yarim_inv(Y, U, XT, XU, R):\n    a, b = _param(U, XT, XU); f = SI[Y] ^ XT\n    return (-b * f * INV[(a + R[:, None] * f) % 256]) % 256\n\ndef boglash(holat, X, R, p):\n    """holat, X: (N,32) birliklar; R: (N,) juft bayt. Natija: (N,32) birliklar."""\n    H = juftlash(holat, p).copy(); Xb = juftlash(X, p); L, Rt = yarimlar(H.shape[1])\n    H[:, Rt] = _yarim(H[:, Rt], H[:, L], Xb[:, Rt], Xb[:, L], R)\n    H[:, L] = _yarim(H[:, L], H[:, Rt], Xb[:, L], Xb[:, Rt], R)\n    H[:, Rt] = _yarim(H[:, Rt], H[:, L], Xb[:, Rt], Xb[:, L], R)\n    return ajratish(H, p)\ndef boglash_inv(Y, X, R, p):\n    H = juftlash(Y, p).copy(); Xb = juftlash(X, p); L, Rt = yarimlar(H.shape[1])\n    H[:, Rt] = _yarim_inv(H[:, Rt], H[:, L], Xb[:, Rt], Xb[:, L], R)\n    H[:, L] = _yarim_inv(H[:, L], H[:, Rt], Xb[:, L], Xb[:, Rt], R)\n    H[:, Rt] = _yarim_inv(H[:, Rt], H[:, L], Xb[:, Rt], Xb[:, L], R)\n    return ajratish(H, p)\n\nRC = np.array([[int(S[(16 * j + i) % 256]) for i in range(32)] for j in range(11)])   # raund konstantalari\n\ndef iteration_b(holat, blok, ke, k, c, sbz=True, rounds=10, trace=None):\n    """Yangi variant: Qo‘sh -> Bog‘lash; yutish: Bog‘lash(holat, blok) [-> SBZ(holat, blok)];\n    kalit jadvali holatdan mustaqil: kₑ ← (SurKalit(kₑ) ⊕ RC_j) ∨ 1."""\n    p = c.p; R = juftlash(k, p)[:, 0] & 0xFE; m = p - 1\n    def yut(h):\n        h = boglash(h, blok, R, p)\n        return bzf(h, blok, c) if sbz else h\n    ke = ke | 1\n    holat = yut(holat)\n    for j in range(1, rounds + 1):\n        holat = aralash(holat, ke, c)\n        holat = boglash(holat, ke, R, p)\n        holat = surholat(holat); ke = (surkalit(ke) ^ (RC[j] & m)) | 1\n        holat = yut(holat) if j == 8 else boglash(holat, ke, R, p)\n        if j == 9: holat = yut(holat)\n        if trace is not None: trace.append((j, holat.copy(), ke.copy()))\n    return holat, ke\n\ndef xesh_b(msgs, keys, p, sbz=True):\n    """msgs: (N,L) bayt (L blok uzunligiga karrali), keys: (N,32 yoki 16) bayt."""\n    c = Cfg(p, \'sbz\'); N, L = msgs.shape; bl = 32 if p == 256 else 16; nb = L // bl\n    blocks = [msgs[:, i * bl:(i + 1) * bl] for i in range(nb)]\n    ny = np.zeros((N, bl), dtype=np.int64)\n    for n in range(N):\n        v = sum(int.from_bytes(bytes(b[n].tolist()), \'big\') for b in blocks) % 2 ** (8 * bl); ny[n] = list(v.to_bytes(bl, \'big\'))\n    lb = np.tile(np.array(list((L * 8).to_bytes(bl, \'big\'))), (N, 1))\n    kir = [to_units(b, p) for b in blocks] + [to_units(lb, p), to_units(ny, p)]\n    k = to_units(keys, p); ke = k.copy(); holat = kir[0]\n    for i in range(1, len(kir)): holat, ke = iteration_b(holat, kir[i], ke, k, c, sbz)\n    return holat\n',
}
for _nom, _matn in _MODULLAR.items():
    if importlib.util.find_spec(_nom) is not None and _nom not in sys.modules:
        print(f"{_nom}.py — papkadagi fayldan olinadi"); continue
    _m = types.ModuleType(_nom); _m.__file__ = f"<{_nom}>"
    sys.modules[_nom] = _m; exec(compile(_matn, f"<{_nom}>", "exec"), _m.__dict__)
    print(f"{_nom} — daftar ichidan yuklandi")


algoritm — daftar ichidan yuklandi
komponentlar — daftar ichidan yuklandi
boglash — daftar ichidan yuklandi


In [2]:
import numpy as np, json, os, sys, time, shutil, subprocess
from algoritm import qosh, Cfg, KAT_XABAR, KAT_KALIT, to_units
from boglash import boglash, boglash_inv
CHAQIRUV = 1_000_000      # C o‘lchovida bitta takrordagi chaqiruvlar soni
TAKROR, TURLAR = 3, 4     # takrorlar (eng yaxshisi olinadi) va aralash tartibdagi turlar soni
NATIJA = {}
rng = np.random.default_rng(1)
for p in (256, 16):
    N = 2000; h = rng.integers(0, p, (N, 32)); x = rng.integers(0, p, (N, 32)); R = rng.integers(0, 128, N) * 2
    print(f"p={p}: Bog‘lash teskarilanuvchi (2000 tasodifiy kirish):", bool((boglash_inv(boglash(h, x, R, p), x, R, p) == h).all()))

p=256: Bog‘lash teskarilanuvchi (2000 tasodifiy kirish): True
p=16: Bog‘lash teskarilanuvchi (2000 tasodifiy kirish): True


## 1. C realizatsiyasi va to‘g‘rilik tekshiruvi

In [3]:
C_MANBA = '/* O\'zMSt 285:2024, 1-algoritm: C realizatsiyasi (tezlikni o\'lchash uchun).\n   Variantlar: p = 256 yoki 16; bz = 0 (BaytZichlash) yoki 1 (SBZ);\n   toq = 1 — barcha kₑ elementlari toq (2-tavsiya); sqatlam = 1 — raund ichidagi S-box qatlami (3-tavsiya). */\n#include <stdint.h>\n#include <string.h>\n#include <stdio.h>\n#include <stdlib.h>\n#include <time.h>\n\nstatic const uint8_t ZICH[256] = {\n 5,13,6,11,1,10,15,8,0,4,7,9,2,12,3,14, 8,7,2,14,15,3,11,6,1,12,13,10,5,4,9,0,\n 14,2,13,4,12,7,1,11,6,9,0,5,3,10,8,15, 0,14,9,12,3,13,7,4,15,6,5,1,11,2,10,8,\n 3,10,7,2,4,12,9,1,14,13,15,8,0,5,11,6, 2,3,1,8,0,14,5,9,12,11,6,7,10,15,13,4,\n 10,4,14,15,9,5,8,2,11,0,1,3,12,6,7,13, 11,9,10,1,6,4,13,15,3,5,14,0,8,7,2,12,\n 1,0,3,7,13,11,10,12,9,14,4,6,15,8,5,2, 4,8,11,9,14,6,2,5,10,3,12,15,7,13,0,1,\n 9,12,15,0,2,1,14,10,5,8,11,13,4,3,6,7, 6,11,8,13,7,9,0,3,4,15,10,2,14,1,12,5,\n 15,1,0,5,10,8,3,7,13,2,9,12,6,14,4,11, 12,5,4,10,11,2,6,13,8,7,3,14,1,0,15,9,\n 7,15,12,6,5,0,4,14,2,10,8,11,13,9,1,3, 13,6,5,3,8,15,12,0,7,1,2,4,9,11,14,10};\nstatic uint8_t SB[256];\nstatic const uint8_t PRESENT[16] = {0xC,5,6,0xB,9,0,0xA,0xD,3,0xE,0xF,8,4,7,1,2};\nstatic uint32_t INV256[256], INV16[16];\n\nstatic void init_tables(void) {\n    uint8_t p = 1, q = 1;\n    do {\n        p = p ^ (uint8_t)(p << 1) ^ ((p & 0x80) ? 0x1B : 0);\n        q ^= q << 1; q ^= q << 2; q ^= q << 4; if (q & 0x80) q ^= 0x09;\n        uint8_t x = q ^ (uint8_t)((q << 1) | (q >> 7)) ^ (uint8_t)((q << 2) | (q >> 6)) ^ (uint8_t)((q << 3) | (q >> 5)) ^ (uint8_t)((q << 4) | (q >> 4));\n        SB[p] = x ^ 0x63;\n    } while (p != 1);\n    SB[0] = 0x63;\n    for (uint32_t x = 1; x < 256; x += 2) for (uint32_t y = 1; y < 256; y += 2) if (((x * y) & 255) == 1) INV256[x] = y;\n    for (uint32_t x = 1; x < 16; x += 2) for (uint32_t y = 1; y < 16; y += 2) if (((x * y) & 15) == 1) INV16[x] = y;\n}\n\ntypedef struct { uint32_t p, m; const uint32_t *inv; int bz, toq, sq; } cfg_t;\n\nstatic inline uint32_t step(uint32_t x, uint32_t A, uint32_t B, uint32_t R, const cfg_t *c) {\n    uint32_t d = (B + R * x) & c->m;\n    return (0u - A * x * c->inv[d]) & c->m;\n}\nstatic inline uint32_t prododd(const uint32_t *a, const cfg_t *c) {\n    uint32_t r = 1; for (int i = 0; i < 32; i++) r = (r * (a[i] | 1)) & c->m; return r;\n}\nstatic inline uint32_t dK(uint32_t P, uint32_t Jo, const cfg_t *c) {\n    uint32_t t = (Jo + 1) & c->m; return (P == t || P == ((0u - t) & c->m)) ? 10 : 0;\n}\nstatic void qosh(uint32_t *h, uint32_t *hn, uint32_t R, const cfg_t *c) {\n    uint32_t m = c->m, J1h = 0, J1hn = 0, h1[32], hn1[32];\n    for (int i = 0; i < 32; i++) { J1h += h[i]; J1hn += hn[i]; }\n    J1h &= m; J1hn &= m;\n    uint32_t Ph = prododd(h, c), Phn = prododd(hn, c);\n    uint32_t K1h = (Ph + dK(Ph, J1hn, c)) & m, K1hn = (Phn + dK(Phn, J1h, c)) & m;\n    uint32_t dAh = (J1hn & 1) ? 0 : 1, dAhn = (J1h & 1) ? 0 : 1;\n    for (int i = 0; i < 32; i++) {\n        h1[i]  = step(h[i],  (J1hn + 2 * hn[i] + dAh) & m, (K1h - 2 * h[i]) & m, R, c);\n        hn1[i] = step(hn[i], (J1h + 2 * h[i] + dAhn) & m, (K1hn - 2 * hn[i]) & m, R, c);\n    }\n    uint32_t J2h = 0, J2hn = 0;\n    for (int i = 0; i < 32; i++) { J2h += h1[i]; J2hn += hn1[i]; }\n    J2h &= m; J2hn &= m;\n    uint32_t P2h = prododd(h1, c), P2hn = prododd(hn1, c);\n    uint32_t K2h = (P2h + dK(P2h, J2hn, c)) & m, K2hn = (P2hn + dK(P2hn, J2h, c)) & m;\n    uint32_t dA2h = ((J1hn + J2hn) & 1) ? 0 : 1, dA2hn = ((J1h + J2h) & 1) ? 0 : 1;\n    uint32_t KKh = (K1h * K2h) & m, KKhn = (K1hn * K2hn) & m;\n    for (int i = 0; i < 32; i++) {\n        uint32_t A2 = (J1hn + J2hn + 2 * hn1[i] + dA2h) & m; if (!A2) A2 = 1;\n        uint32_t B2 = (KKh - 2 * h1[i]) & m;\n        uint32_t x2 = step(h1[i], A2, B2, R, c); if (!x2) x2 = (A2 + B2) & m;\n        uint32_t A2n = (J1h + J2h + 2 * h1[i] + dA2hn) & m; if (!A2n) A2n = 1;\n        uint32_t B2n = (KKhn - 2 * hn1[i]) & m;\n        uint32_t y2 = step(hn1[i], A2n, B2n, R, c); if (!y2) y2 = (A2n + B2n) & m;\n        h[i] = step(x2, A2, B2, R, c); hn[i] = step(y2, A2n, B2n, R, c);\n    }\n}\n\n/* ---------- Bog\'lash (ikkala rejim uchun yagona, bayt darajasida) ---------- */\nstatic const int L256[16] = {0,1,2,3, 8,9,10,11, 16,17,18,19, 24,25,26,27};\nstatic const int R256[16] = {4,5,6,7, 12,13,14,15, 20,21,22,23, 28,29,30,31};\nstatic const int L128[8]  = {0,1, 4,5, 8,9, 12,13};\nstatic const int R128[8]  = {2,3, 6,7, 10,11, 14,15};\nstatic inline void yarim(uint8_t *H, const int *T, const int *U, int m, const uint8_t *X, uint32_t R) {\n    uint32_t J = 0, K = 1;\n    for (int i = 0; i < m; i++) { uint32_t u = H[U[i]]; J += u; K *= (u | 1); }\n    J &= 255; K &= 255;\n    uint32_t u0 = H[U[0]];\n    for (int i = 0; i < m; i++) {\n        uint32_t ui = H[U[i]], un = (i + 1 < m) ? H[U[i + 1]] : u0, xt = X[T[i]], x = H[T[i]];\n        uint32_t a = 2u * SB[ui ^ X[U[i]] ^ J] + 1, b = 2u * SB[un ^ xt ^ K] + 1;\n        uint32_t f = (0u - a * x * INV256[(b + R * x) & 255]) & 255;\n        H[T[i]] = SB[f ^ xt];\n    }\n}\nstatic void boglash(uint32_t *h, const uint32_t *x, uint32_t R, int p) {\n    uint8_t H[32], X[32];\n    if (p == 256) {\n        for (int i = 0; i < 32; i++) { H[i] = (uint8_t)h[i]; X[i] = (uint8_t)x[i]; }\n        yarim(H, R256, L256, 16, X, R); yarim(H, L256, R256, 16, X, R); yarim(H, R256, L256, 16, X, R);\n        for (int i = 0; i < 32; i++) h[i] = H[i];\n    } else {\n        for (int j = 0; j < 16; j++) { H[j] = (uint8_t)((h[2*j] << 4) | h[2*j+1]); X[j] = (uint8_t)((x[2*j] << 4) | x[2*j+1]); }\n        yarim(H, R128, L128, 8, X, R); yarim(H, L128, R128, 8, X, R); yarim(H, R128, L128, 8, X, R);\n        for (int j = 0; j < 16; j++) { h[2*j] = H[j] >> 4; h[2*j+1] = H[j] & 15; }\n    }\n}\nstatic double now(void) { return (double)clock() / CLOCKS_PER_SEC; }\nint main(int argc, char **argv) {\n    init_tables();\n    static const uint8_t KEY[32] = {0xc8,0x6e,0xd4,0xb2,0x75,0x6d,0xf1,0xcd,0x6b,0x6e,0xca,0x81,0xd4,0x41,0x80,0x92,\n                                    0x21,0xab,0x81,0xbb,0xa0,0xc7,0xc1,0x50,0x44,0xbe,0x8f,0xeb,0x93,0x4f,0x95,0xa2};\n    const char *M = "funksiyaxeshirovaniyainformasionnayatexnologiyakriptograficheska";\n    if (argc > 1 && !strcmp(argv[1], "test")) {      /* to\'g\'rilikni tekshirish: A ilova ma\'lumotlari */\n        for (int p = 256; p >= 16; p /= 16) {\n            cfg_t c = {(uint32_t)p, (uint32_t)p - 1, p == 256 ? INV256 : INV16, 0, 0, 0};\n            uint32_t h[32], x[32], h2[32], x2[32];\n            for (int i = 0; i < 32; i++) {\n                if (p == 256) { h[i] = (uint8_t)M[i]; x[i] = (uint8_t)M[32 + i]; }\n                else { h[i] = (i % 2) ? ((uint8_t)M[i/2] & 15) : ((uint8_t)M[i/2] >> 4); x[i] = (i % 2) ? ((uint8_t)M[16+i/2] & 15) : ((uint8_t)M[16+i/2] >> 4); }\n            }\n            uint32_t Rq = p == 256 ? KEY[0] : (KEY[0] >> 4), Rb = KEY[0] & 0xFE;\n            memcpy(h2, h, sizeof h); memcpy(x2, x, sizeof x); qosh(h2, x2, Rq, &c);\n            printf("qosh %d ", p); for (int i = 0; i < 32; i++) printf(p == 256 ? "%02x" : "%x", h2[i]); printf("\\n");\n            memcpy(h2, h, sizeof h); boglash(h2, x, Rb, p);\n            printf("boglash %d ", p); for (int i = 0; i < 32; i++) printf(p == 256 ? "%02x" : "%x", h2[i]); printf("\\n");\n        }\n        return 0;\n    }\n    /* tezlik: argv = amal(qosh|boglash) p chaqiruvlar_soni takror */\n    int isq = !strcmp(argv[1], "qosh"), p = atoi(argv[2]); long n = atol(argv[3]); int rep = atoi(argv[4]);\n    cfg_t c = {(uint32_t)p, (uint32_t)p - 1, p == 256 ? INV256 : INV16, 0, 0, 0};\n    uint32_t h[32], x[32]; double best = 1e9; uint32_t chk = 0;\n    for (int r = 0; r < rep; r++) {\n        for (int i = 0; i < 32; i++) { h[i] = (i * 37 + 11) & c.m; x[i] = (i * 91 + 5) & c.m; }\n        double t = now();\n        for (long it = 0; it < n; it++) {\n            if (isq) { qosh(h, x, p == 256 ? 200 : 12, &c); }          /* x ham o\'zgaradi (kₑ kabi) */\n            else { boglash(h, x, 200, p); x[it & 31] ^= h[(it * 7) & 31]; }\n        }\n        t = now() - t; if (t < best) best = t; chk ^= h[0] ^ x[1];\n    }\n    printf("%.6f %u\\n", best, chk);\n    return 0;\n}\n'

In [4]:
open('qb.c', 'w', encoding='utf-8').write(C_MANBA)
EXE = os.path.abspath('qb.exe' if sys.platform.startswith('win') else 'qb')
CC = next((c for c in ('gcc', 'cc', 'clang') if shutil.which(c)), None); C_BOR = False
if CC:
    r = subprocess.run([CC, '-O2', '-o', EXE, 'qb.c'], capture_output=True, text=True); C_BOR = r.returncode == 0
    print("Kompilyator:", CC, "— muvaffaqiyatli" if C_BOR else "— xato:\n" + r.stderr)
else: print("C kompilyatori topilmadi — C o‘lchovi o‘tkazib yuboriladi.")
if C_BOR:
    out = subprocess.run([EXE, 'test'], capture_output=True, text=True).stdout
    M = np.array(list(KAT_XABAR))
    for p in (256, 16):
        if p == 256: h, x, Rq = M[:32][None], M[32:][None], np.array([KAT_KALIT[0]])
        else: h, x, Rq = to_units(M[:16][None], 16), to_units(M[16:32][None], 16), np.array([KAT_KALIT[0] >> 4])
        fmt = (lambda a: ''.join(f'{int(v):02x}' for v in a)) if p == 256 else (lambda a: ''.join(f'{int(v):x}' for v in a))
        q = fmt(qosh(h, x, Rq, Cfg(p, 'std'))[0][0]); b = fmt(boglash(h, x, np.array([KAT_KALIT[0] & 0xFE]), p)[0])
        print(f"p={p}: Qo‘sh C = Python: {f'qosh {p} {q}' in out};  Bog‘lash C = Python: {f'boglash {p} {b}' in out}")
    print("(256 bitli rejimdagi Qo‘sh natijasi standartning A ilovasidagi birinchi Qo‘sh natijasi bilan mos: 8a ff 7e 0d …)")

Kompilyator: gcc — muvaffaqiyatli
p=256: Qo‘sh C = Python: True;  Bog‘lash C = Python: True
p=16: Qo‘sh C = Python: True;  Bog‘lash C = Python: True
(256 bitli rejimdagi Qo‘sh natijasi standartning A ilovasidagi birinchi Qo‘sh natijasi bilan mos: 8a ff 7e 0d …)


In [5]:
if C_BOR:
    res = {}
    for tur in range(TURLAR):
        for p in (256, 16):
            for op in ('qosh', 'boglash'):
                t = float(subprocess.run([EXE, op, str(p), str(CHAQIRUV), str(TAKROR)], capture_output=True, text=True).stdout.split()[0])
                res.setdefault(f'{op}_{p}', []).append(t / CHAQIRUV * 1e9)
    NATIJA['C_ns'] = {k: round(min(v), 1) for k, v in res.items()}; NATIJA['C_mediana'] = {k: round(float(np.median(v)), 1) for k, v in res.items()}
    print("ns / chaqiruv:   eng yaxshi (mediana)")
    for p in (256, 16):
        q, b = NATIJA['C_ns'][f'qosh_{p}'], NATIJA['C_ns'][f'boglash_{p}']
        print(f"  p={p:3d}: Qo‘sh {q:7.1f} ({NATIJA['C_mediana'][f'qosh_{p}']}),  Bog‘lash {b:7.1f} ({NATIJA['C_mediana'][f'boglash_{p}']}),  "
              f"Bog‘lash {q/b:.2f} marta tez (mediana bo‘yicha {NATIJA['C_mediana'][f'qosh_{p}']/NATIJA['C_mediana'][f'boglash_{p}']:.2f})")
    print("Bir blokda 22 ta chaqiruv: tejalgan vaqt ≈", {p: f"{22*(NATIJA['C_ns'][f'qosh_{p}']-NATIJA['C_ns'][f'boglash_{p}'])/1000:.2f} mkS" for p in (256, 16)})

ns / chaqiruv:   eng yaxshi (mediana)
  p=256: Qo‘sh   263.8 (303.3),  Bog‘lash   130.4 (137.3),  Bog‘lash 2.02 marta tez (mediana bo‘yicha 2.21)
  p= 16: Qo‘sh   255.1 (290.0),  Bog‘lash    96.6 (99.8),  Bog‘lash 2.64 marta tez (mediana bo‘yicha 2.91)
Bir blokda 22 ta chaqiruv: tejalgan vaqt ≈ {256: '2.93 mkS', 16: '3.49 mkS'}


## 2. Python (numpy) realizatsiyasi

In [6]:
NATIJA['Python_mks'] = {}
for p in (256, 16):
    for N in (1, 4000):
        h = rng.integers(0, p, (N, 32)); x = rng.integers(0, p, (N, 32)); R = np.full(N, 200 if p == 256 else 12); Rb = np.full(N, 200); rep = 200 if N == 1 else 5
        t0 = time.perf_counter(); [qosh(h, x, R, Cfg(p, 'std')) for _ in range(rep)]; tq = (time.perf_counter() - t0) / rep / N * 1e6
        t0 = time.perf_counter(); [boglash(h, x, Rb, p) for _ in range(rep)]; tb = (time.perf_counter() - t0) / rep / N * 1e6
        NATIJA['Python_mks'][f'{p}_{N}'] = (round(tq, 2), round(tb, 2))
        print(f"p={p:3d}, bir vaqtda {N:4d} ta: Qo‘sh {tq:8.2f} mkS, Bog‘lash {tb:8.2f} mkS  ->  {tq/tb:.2f} marta")

p=256, bir vaqtda    1 ta: Qo‘sh   288.37 mkS, Bog‘lash   196.82 mkS  ->  1.47 marta
p=256, bir vaqtda 4000 ta: Qo‘sh     9.84 mkS, Bog‘lash     3.04 mkS  ->  3.24 marta
p= 16, bir vaqtda    1 ta: Qo‘sh   284.55 mkS, Bog‘lash   147.65 mkS  ->  1.93 marta
p= 16, bir vaqtda 4000 ta: Qo‘sh     8.12 mkS, Bog‘lash     1.72 mkS  ->  4.73 marta


## 3. Amallar soni (bitta chaqiruv)

In [7]:
def amallar_qosh():
    return dict(kopaytirish=128 + 64 * 3 + 2 + 64 * 6, teskari_jadval=192, s_jadval=0, qoshish_xor=128 + 64 * 6 + 64 * 10, shartli_tekshiruv=4 + 64 * 2)
def amallar_boglash(p):
    m = 16 if p == 256 else 8; e = 3 * m                          # 3 yarim qadam
    return dict(kopaytirish=3 * m + 3 * e, teskari_jadval=e, s_jadval=3 * e, qoshish_xor=3 * m + 7 * e + (0 if p == 256 else 64), shartli_tekshiruv=0)
NATIJA['amallar'] = {'qosh': amallar_qosh(), 'boglash_256': amallar_boglash(256), 'boglash_128': amallar_boglash(16)}
for k, v in NATIJA['amallar'].items(): print(f"{k:12s}", v, " jami:", sum(v.values()))
json.dump(NATIJA, open('qosh_boglash_tezlik.json', 'w'), ensure_ascii=False, indent=1)
try:
    import matplotlib.pyplot as plt
    if 'C_ns' in NATIJA:
        fig, ax = plt.subplots(figsize=(6, 3.2)); xs = np.arange(2)
        ax.bar(xs - 0.2, [NATIJA['C_ns']['qosh_256'], NATIJA['C_ns']['qosh_16']], 0.4, label='Qo‘sh')
        ax.bar(xs + 0.2, [NATIJA['C_ns']['boglash_256'], NATIJA['C_ns']['boglash_16']], 0.4, label='Bog‘lash')
        ax.set_xticks(xs); ax.set_xticklabels(['256 bitli rejim', '128 bitli rejim']); ax.set_ylabel('ns / chaqiruv (C, -O2)'); ax.legend(); ax.grid(alpha=.3)
        fig.tight_layout(); fig.savefig('qosh_boglash_tezlik.png', dpi=200); plt.show()
except ImportError: pass
print("Natijalar qosh_boglash_tezlik.json ga yozildi.")

qosh         {'kopaytirish': 706, 'teskari_jadval': 192, 's_jadval': 0, 'qoshish_xor': 1152, 'shartli_tekshiruv': 132}  jami: 2182
boglash_256  {'kopaytirish': 192, 'teskari_jadval': 48, 's_jadval': 144, 'qoshish_xor': 384, 'shartli_tekshiruv': 0}  jami: 768
boglash_128  {'kopaytirish': 96, 'teskari_jadval': 24, 's_jadval': 72, 'qoshish_xor': 256, 'shartli_tekshiruv': 0}  jami: 448
Natijalar qosh_boglash_tezlik.json ga yozildi.
